In [ ]:
import os
import re
import pickle
from datasets import Dataset
import pandas as pd
import json
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from huggingface_hub import login

import torch
from torch.nn.parallel import DataParallel
from torch.cuda.amp import GradScaler, autocast

import random
import numpy as np

#### seed 고정 ####
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
###################

#캐시 제거
import torch
torch.cuda.empty_cache()


import huggingface_hub
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
    EarlyStoppingCallback,  # Importing EarlyStoppingCallback
)

from transformers import TrainerCallback #추가

from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model
)
from trl import SFTTrainer, SFTConfig

# Login to Hugging Face Hub
# huggingface_hub.login(token="put your hf token", add_to_git_credential=True)
login(token="put your hf token")

# Assign variables
# base_path = "/root/pkl/"
# raw_filename = "wholespine_ori_question.pkl"
base_path = "/root/pj_llm/dataset/preprocessed/pkl/"
raw_filename = "wholespine_ori_question.pkl"
# Set models
base_model = "ProbeMedicalYonseiMAILab/medllama3-v20"           # Hugging Face Basic Model
new_model = "ProbeMedicalYonseiMAILab/medllama3-v20-finetuned"  # Fine tuning Model

with open(base_path + raw_filename, 'rb') as f:
    raw_dataset = pickle.load(f)

def create_text_column(example):
    text = f"### Instruction:\n{example['instruction']}\n\nInput:\n{example['input']}\n\n### Response:\n{example['output']}"
    return text

raw_dataset['text'] = raw_dataset.apply(create_text_column, axis=1)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # 일관성을 위해 처음부터 설정
EOS_TOKEN = tokenizer.eos_token

def prompt_eos(example):
    example['text'] = example['text'] + EOS_TOKEN
    return example

raw_dataset = raw_dataset.apply(prompt_eos, axis=1)

# Divide into train, eval and test
'''
The Ratio of
train dataset: 70% / 80%
eval dataset:  15% / 10%
test dataset:  15% / 10%
'''
# Split data to train/eval/test
train_dataset, temp_dataset = train_test_split(raw_dataset, test_size=0.30, stratify=raw_dataset['output'], random_state=42)
eval_dataset, test_dataset = train_test_split(temp_dataset, test_size=0.50, stratify=temp_dataset['output'], random_state=42)
# train_dataset, temp_dataset = train_test_split(raw_dataset, test_size=0.2, stratify=raw_dataset['output'], random_state=42)
# eval_dataset, test_dataset = train_test_split(temp_dataset, test_size=0.5, stratify=temp_dataset['output'], random_state=42)
# Reset index
train_dataset.reset_index(drop=True, inplace=True)
eval_dataset.reset_index(drop=True, inplace=True)
test_dataset.reset_index(drop=True, inplace=True)

# # Convert to Hugging Face Dataset
# train_dataset = Dataset.from_pandas(train_dataset)
# eval_dataset = Dataset.from_pandas(eval_dataset)
# test_dataset = Dataset.from_pandas(test_dataset)
# print(f"Number of entries in train_dataset: {len(train_dataset)}")
# print(f"Number of entries in eval_dataset: {len(eval_dataset)}")
# print(f"Number of entries in test_dataset: {len(test_dataset)}")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [24]:
test_dataset

,input,instruction,output,text
0,\n\nClinical information : Hepatocellular carc...,"Based on the report below, classify a cancer t...",romets,"### Instruction:\nBased on the report below, c..."
1,\nExam : Whole spine MRI with contrast enhance...,"Based on the report below, classify a cancer t...",improved,"### Instruction:\nBased on the report below, c..."
2,Exam : Whole spine MRI with contrast enhanceme...,"Based on the report below, classify a cancer t...",romets,"### Instruction:\nBased on the report below, c..."
3,EXAM: Whole spine MRI with contrast enhancemen...,"Based on the report below, classify a cancer t...",stable,"### Instruction:\nBased on the report below, c..."
4,\nExam: Whole spine MRI with contrast enhancem...,"Based on the report below, classify a cancer t...",romets,"### Instruction:\nBased on the report below, c..."
...,...,...,...,...
100,\n\nEXAM: Whole spine MRI with contrast enhanc...,"Based on the report below, classify a cancer t...",improved,"### Instruction:\nBased on the report below, c..."
101,\n\nEXAM: Whole-spine MRI with contrast enhanc...,"Based on the report below, classify a cancer t...",improved,"### Instruction:\nBased on the report below, c..."
102,Exam: Whole spine contrast enhanced MRI.\nClin...,"Based on the report below, classify a cancer t...",mets,"### Instruction:\nBased on the report below, c..."
103,"\n\nClinical information: Esophageal cancer, s...","Based on the report below, classify a cancer t...",no,"### Instruction:\nBased on the report below, c..."


In [2]:
test_dataset = test_dataset.drop(['instruction', 'text', 'output'], axis=1)

In [25]:
# sequence_length 1024, 4bit quantization full-finetuning
df_1 = pd.read_csv("/root/modelfile/models/sy_original_7_len1024_4bit_fullfinetuning/metrics/20240817_032147/results_20240817_032147.csv")

In [26]:
new_df = pd.concat([df_1, test_dataset], axis=1)

In [27]:
new_df['report'] = new_df['input']
new_df = new_df.drop(['input'], axis=1)

In [28]:
new_df

,correct,predicted,report
0,romets,romets,\n\nClinical information : Hepatocellular carc...
1,improved,improved,\nExam : Whole spine MRI with contrast enhance...
2,romets,romets,Exam : Whole spine MRI with contrast enhanceme...
3,stable,stable,EXAM: Whole spine MRI with contrast enhancemen...
4,romets,romets,\nExam: Whole spine MRI with contrast enhancem...
...,...,...,...
100,improved,improved,\n\nEXAM: Whole spine MRI with contrast enhanc...
101,improved,improved,\n\nEXAM: Whole-spine MRI with contrast enhanc...
102,mets,mets,Exam: Whole spine contrast enhanced MRI.\nClin...
103,no,no,"\n\nClinical information: Esophageal cancer, s..."


In [88]:
print(new_df['report'].iloc[103])



Clinical information: Esophageal cancer, s/p salvage CCRTx with XP and FP (~2020.10)  
  
Exam: Whole spine MRI with contrast enhancement.  
  
Findings:  
No evidence of bone marrow replacing lesion along the spine. 
Probable vertebral hemangioma on T2 vertebral body.
Soft tissue lesion with peripheral enhancement in right erector spinae in L5 level, probable bursitis.
Compression fracture of T5, T7, L3 without bone marrow signal change or contrast enhancement, probably benign old fracture.  
No definite central canal stenosis or neural foraminal stenosis.  
  
Conclusion:  
1. No evidence of metastatic bone lesion.
2. Probable benign old compression fracture of T5, T7 and L3 vertebrae.  
  
Recommend:  
Clinical correlation and follow up MRI.


In [29]:
uncorrect_df = new_df[new_df['correct'] != new_df['predicted']]

In [30]:
uncorrect_df

,correct,predicted,report
13,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
23,improved,stable,\n \nEXAM : Whole spine MRI with contrast enh...
26,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
65,progression,romets,\n\nEXAM : Whole spine MRI with contrast enhan...
70,romets,improved,Exam : Whole spine MRI with contrast enhanceme...
74,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
93,romets,improved,EXAM : Whole spine MRI with contrast enhanceme...
95,romets,no,\n\nEXAM: Whole spine MRI with enhancement. \n...


In [32]:
romets_uncorrect_df = uncorrect_df[uncorrect_df['correct']=='romets']

romets_uncorrect_improved_df = romets_uncorrect_df[romets_uncorrect_df['predicted']=='improved']
romets_uncorrect_no_df = romets_uncorrect_df[romets_uncorrect_df['predicted']!='improved']

In [33]:
improved_uncorrect_df = uncorrect_df[uncorrect_df['correct']=='improved']
progression_uncorrect_df = uncorrect_df[uncorrect_df['correct']=='progression']

## romets를 improved라고 잘못 예측하는 데이터를 분석

In [34]:
romets_uncorrect_improved_df

,correct,predicted,report
13,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
26,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
70,romets,improved,Exam : Whole spine MRI with contrast enhanceme...
74,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
93,romets,improved,EXAM : Whole spine MRI with contrast enhanceme...


In [35]:
romets_uncorrect_improved_list = romets_uncorrect_improved_df.report.to_list()

In [43]:
print(romets_uncorrect_improved_list[0])

EXAM: Whole spine MRI with contrast enhancement and metal artifact reduction.
Clinical information: Bilateral hand numbness, s/p decompressive laminectomy and posterior fixation, s/p corpectomy and anterior interbody fixation due pathologic fracture C7 (Breast Cancer), s/p Radiotherapy.
Findings:
s/p Decompressive laminectomy and posterior fixation, C4-5-6-T1-2.
s/p Anterior cervical corpectomy and fusion C6-T1.
Swelling and T2 hyperintensity with enhancement in posterior back muscle and subcutaneous fat layer of cervical to upper thoracic level.
• Epidural extension and encroachment of both foraminal canal at C6/7-C7/T1.
Bone marrow signal change and enhancement in C6 and T1.
Diffuse marrow signal changes including clivus and cervicothoracic vertebrae.
• Clinical significance indeterminate.
Conclusions:
s/p Decompressive laminectomy and posterior fixation, C4-5-6-T1-2.
s/p Anterior cervical corpectomy and fusion C6-T1.
Decreased epidural extent compared with 2021-12-16 MRI.
R/O Remain

In [45]:
print(romets_uncorrect_improved_list[1])

EXAM: Whole spine MRI with contrast enhancement and metal artifact reduction.
Clinical information: Upper limb weakness, s/p decompressive laminectomy and posterior fixation, s/p corpectomy and anterior interbody fixation due pathologic fracture T2 (Osteosarcoma), s/p Radiotherapy.
Findings:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Swelling and T2 hyperintensity with enhancement in posterior back muscle and subcutaneous fat layer of thoracic level.
• Epidural extension and encroachment of both foraminal canal at T2/3.
Bone marrow signal change and enhancement in T2 and T3.
Diffuse marrow signal changes including thoracic vertebrae.
• Clinical significance indeterminate.
Conclusions:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Decreased epidural extent compared with 2021-12-16 MRI.
R/O Remained lesion, T2 and T3.
Recommendation:
Clinical co

In [46]:
print(romets_uncorrect_improved_list[2])

Exam : Whole spine MRI with contrast enhancement.  
Clinical information : RCC with multiple metastasis.      
COMPARISON : 2021-04-30 MRI.  
  
Findings :
Newly developed marrow replacing enhancing nodular lesion in the upper endplate of L2 right body. Lesion shows peripheral enhanement with internal  nonenhancement, peripheral low signal intensity rim. Schemorl`s node is suggested rather than metastasis.

Decreased peripheral enhancement of multiple marrow replacing lesions in T2 and T4, L4, right ilium and left femoral neck within scanned range.
No change of fatty marrow change in upper T- and lower L-spine to sacrum, suggestive of RTx-induced change.
 
 
Conclusion :
1. Newly devloped lesion, L2 body.
    - R/O Schmorl`s node
      - DDx. Metastasis.
2. Decreased enhancement of known multiple bone metastasis.


Recommendation : 
Clinical correlation and MRI short term follow up.


In [47]:
print(romets_uncorrect_improved_list[3])

EXAM: Whole spine MRI with contrast enhancement and metal artifact reduction.
Clinical information: Upper back pain, s/p decompressive laminectomy and posterior fixation, s/p corpectomy and anterior interbody fixation due pathologic fracture T5 (Lung Cancer), s/p Radiotherapy.
Findings:
s/p Decompressive laminectomy and posterior fixation, T3-4-5-T7-8.
s/p Anterior thoracic corpectomy and fusion T4-6.
Swelling and T2 hyperintensity with enhancement in posterior back muscle and subcutaneous fat layer of thoracic level.
• Epidural extension and encroachment of both foraminal canal at T5/6.
Bone marrow signal change and enhancement in T5 and T6.
Diffuse marrow signal changes including thoracic vertebrae.
• Clinical significance indeterminate.
Conclusions:
s/p Decompressive laminectomy and posterior fixation, T3-4-5-T7-8.
s/p Anterior thoracic corpectomy and fusion T4-6.
Decreased epidural extent compared with 2021-12-16 MRI.
R/O Remained lesion, T5 and T6.
Recommendation:
Clinical correla

In [48]:
print(romets_uncorrect_improved_list[4])

EXAM : Whole spine MRI with contrast enhancement. 
 
Clinical information : Lung cancer with brain, bone metastases. s/p intramedullary metastatic tumor removal (2022.2.7) s/p RTx on T11-L1. 
 
COMPARISON : Whole spine MRI taken on 2022.2.4. 
 
FINDINGS :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12 levels. 
T2 hyperintensity without enhancement in anterior aspect of conus medullaris at T11, T12 levels. 
- Focal pachymeningeal and leptomeningeal enhancement, subcutaneous T2 hyperintensity with enhancement at T11, T12 levels. 
No definite bone marrow replacing masses in whole spines. 
 
Central disc protrusion at T1/2, L2/3. 
Several enhancing small nodules in cerebellum and occipital lobe of cerebrum. 
R/O Brain metastases, refer to brain MRI taken on 2022.6.13. 
 
CONCLUSION :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12.
s/p Radiotherapy. 
- Focal pachymeningeal and leptomeningeal enhancement.
- Clinical significance i

In [81]:
print(romets_uncorrect_no_df['report'].to_list()[0])



EXAM: Whole spine MRI with enhancement. 
Clinical information: 
Colon cancer (15 YA). 
Pancreas cancer, s/p distal pancreatectomy and CTx. 
 
Findings: 
Compression fracture with marrow edema and enhancement at T3 and T4. 
R/O Schmorl's node at inferior endplate of L4. 
Degenerative spondylolisthesis at L4 over L5. 
Bulging disc at C3/4 through C6/7. 
Unremarkable finding of spinal cord. 
 
Conclusion: 
Recent compression fracture, T3 and T4. 
R/O Schmorl's node at inferior endplate of L4 rather than metastatic bone lesion. 

Recommendation: 
Clinical correlation and follow up. 



### romets를 romets로 잘 맞춘 사례도 보기

In [56]:
correct_df = new_df[new_df['correct'] == new_df['predicted']]
romets_correct_df = correct_df[correct_df['correct']=="romets"]

In [59]:
romets_correct_df

,correct,predicted,report
0,romets,romets,\n\nClinical information : Hepatocellular carc...
2,romets,romets,Exam : Whole spine MRI with contrast enhanceme...
4,romets,romets,\nExam: Whole spine MRI with contrast enhancem...
25,romets,romets,\n \nEXAM: Whole spine MRI with contrast contr...
30,romets,romets,EXAM : Whole spine MRI with contrast enhanceme...
34,romets,romets,EXAM: Whole spine contrast enhanced MRI\nThe p...
38,romets,romets,\nExam: Whole-spine MRI with contrast enhancem...
50,romets,romets,EXAM : Whole spine MRI with contrast enhanceme...
55,romets,romets,\nExam : Whole spine MRI with contrast enhance...
58,romets,romets,\n\nExam : Whole spine MRI with contrast enhan...


In [52]:
print(new_df['report'].loc[0]) #romets를 romets라고 잘 맞춘 사례



Clinical information : Hepatocellular carcinoma. 
EXAM : MRI Whole-spine with contrast enhancement.   
 
FINDINGS : 
Enhancing lesions in C3 and T10 vertebral body and right pedicle, r/o metastases.  
Multilevel disc bulging/herniation at cervical and lumbar spines.  
 
CONCLUSION : 
R/O Metastases in C3 and T10.  
 
RECOMMENDATION : 
follow-up.  



In [54]:
print(new_df['report'].loc[2]) #romets를 romets라고 잘 맞춘 사례

Exam : Whole spine MRI with contrast enhancement.
Clinical information : R/O Multiple myeloma. s/p thyroid cancer.

Findings :
Multiple subtle enhancing nodular lesion with with T1 hyposignal intensity at C-, T-, L-spine, sacrum and both pelvic bones.
Compression fracture of T7.
No significant spinal canal stenosis or signal change of the spinal cord.
Multiple disk degneration of cervical and lumbar spine.


Conclusion :
1. R/O Multiple bone metastases
   - DDx. Multiple myeloma.
2. Pathologic fracture, T7.
3. Multiple disk degeneration.


Recommendation : 
Clinical correlation.


In [53]:
print(new_df['report'].loc[4]) #romets를 romets라고 잘 맞춘 사례


Exam: Whole spine MRI with contrast enhancement.  
Clinical information: s/p tumor removal involving C4 body (mesenchymal chondrosarcoma, total corpectomy) and anterior interbody fusion with discectomy, C3-5 (2006), R/O lung cancer.  
Comparison: 2018-05-17 C-spine MRI.  
 
FINDING: 
s/p Anterior interbody fusion, C3-4-5.  
Focal bone marrow replacing lesion with contrast enhancement at C4 lamina, bilaterally.
Focal enhancement at C6 vertebral body, without remarkable bone marrow signal change on T1WI and T2WI.  
Disc bulging at C5/6 and C6/7, without spinal cord signal change.  
 
Ossification of the ligamentum flavum at T7.   
Old compression fracture of T8 and L2.   
 
Multilevel asymmetric disc bulging, ligamentum flavum thickening and facet joint hypertrophy of lumbar spine.  
 - mild central canal stenosis at L2/3.  
 - severe central canal stenosis and right neural foraminal stenosis at L3/4.  
 - severe central canal stenosis at L3/4 and L4/5.  
 - left neural foraminal stenos

In [55]:
print(new_df['report'].loc[25]) #romets를 romets라고 잘 맞춘 사례


 
EXAM: Whole spine MRI with contrast contrast enhancement.  
Clinical information : Back pain, Ascending colon cancer, adenoca Liver metastasis, multiple, MD, s/p FOLFOX with EMEND(21.01.07 -), last 21.07.09-  
 
FINDINGS:  
Sacralization of L5.  
Bone marrow replacing lesion at L3 anterior body with mild enhancement.  
R/O Perineural cysts at S1 level.  
 
IMPRESSION:  
R/O Bone metastasis at L3 anterior body.  
 
Recommendation: Clinical correlation.


In [69]:
print(romets_correct_df['report'].loc[4])


Exam: Whole spine MRI with contrast enhancement.  
Clinical information: s/p tumor removal involving C4 body (mesenchymal chondrosarcoma, total corpectomy) and anterior interbody fusion with discectomy, C3-5 (2006), R/O lung cancer.  
Comparison: 2018-05-17 C-spine MRI.  
 
FINDING: 
s/p Anterior interbody fusion, C3-4-5.  
Focal bone marrow replacing lesion with contrast enhancement at C4 lamina, bilaterally.
Focal enhancement at C6 vertebral body, without remarkable bone marrow signal change on T1WI and T2WI.  
Disc bulging at C5/6 and C6/7, without spinal cord signal change.  
 
Ossification of the ligamentum flavum at T7.   
Old compression fracture of T8 and L2.   
 
Multilevel asymmetric disc bulging, ligamentum flavum thickening and facet joint hypertrophy of lumbar spine.  
 - mild central canal stenosis at L2/3.  
 - severe central canal stenosis and right neural foraminal stenosis at L3/4.  
 - severe central canal stenosis at L3/4 and L4/5.  
 - left neural foraminal stenos

### 정답 맞춘/못 맞춘 데이터의 구조 차이를 분석

In [71]:
import re

def is_text_divided_into_paragraphs(text):
    # 연속된 두 개 이상의 줄바꿈 문자(또는 다른 공백 문자)를 기준으로 텍스트를 나눕니다.
    paragraphs = re.split(r'\n\s*\n+', text)
    
    # 문단의 수가 1보다 크다면 True, 그렇지 않다면 False 반환
    return len(paragraphs) > 1

In [73]:
#정답 맞춘 데이터
[is_text_divided_into_paragraphs(t) for t in romets_correct_df['report']]

[True, True, True, True, True, True, True, True, True, True, True]

In [82]:
# 정답 못 맞춘 데이터-improved라고 예측한 데이터
[is_text_divided_into_paragraphs(t) for t in romets_uncorrect_improved_df['report']]

[False, False, True, False, True]

In [83]:
#정답 못 맞춘 데이터 - no라고 예측한 데이터
[is_text_divided_into_paragraphs(t) for t in romets_uncorrect_no_df['report']]

[True]

# Prompt Engineering 이후 결과에서 틀린 데이터 분석하기

In [3]:
df_pe = pd.read_csv("/root/codes/promt_engineering/results_prompt/sy/combined_results_basic_baseline prompt and diagnosis process by doctor in more details and CoT and RO insert_20240818_123302.csv")

In [9]:
new_df2 = pd.concat([df_pe, test_dataset], axis=1)

In [10]:
new_df2 = new_df2[['correct','predicted','input']]

In [12]:
new_df2['report'] = new_df2['input']
new_df2 = new_df2.drop(['input'],axis=1)
new_df2

,correct,predicted,report
0,romets,romets,\n\nClinical information : Hepatocellular carc...
1,improved,improved,\nExam : Whole spine MRI with contrast enhance...
2,romets,romets,Exam : Whole spine MRI with contrast enhanceme...
3,stable,stable,EXAM: Whole spine MRI with contrast enhancemen...
4,romets,romets,\nExam: Whole spine MRI with contrast enhancem...
...,...,...,...
100,improved,improved,\n\nEXAM: Whole spine MRI with contrast enhanc...
101,improved,improved,\n\nEXAM: Whole-spine MRI with contrast enhanc...
102,mets,mets,Exam: Whole spine contrast enhanced MRI.\nClin...
103,no,no,"\n\nClinical information: Esophageal cancer, s..."


In [13]:
pe_uncorrect = new_df2[new_df2['correct'] != new_df2['predicted']]
pe_uncorrect

,correct,predicted,report
24,stable,romets,\nExam: Whole spine MRI with contrast enhancem...
26,romets,improved,EXAM: Whole spine MRI with contrast enhancemen...
65,progression,romets,\n\nEXAM : Whole spine MRI with contrast enhan...
70,romets,improved,Exam : Whole spine MRI with contrast enhanceme...
93,romets,improved,EXAM : Whole spine MRI with contrast enhanceme...


### romets를 improved로 잘못 예측한 것 (romets는 악성종양이 재발되었음을 의미, improved는 환자의 상태나 증상이 좋아졌음을 의미함)

In [25]:
print(pe_uncorrect['report'].loc[26]) 

#R/O"는 "Rule Out"의 약자로, 특정 질병이나 상태의 가능성을 배제하거나 확인하기 위해 추가 검사가 필요함을 의미
# 우리 프롬프트에는 그냥 R/O가 romets와 관련이 있다고 적어두었는데, 모델에 R/O의 의미를 알려줘야 할 듯 합니다.

EXAM: Whole spine MRI with contrast enhancement and metal artifact reduction.
Clinical information: Upper limb weakness, s/p decompressive laminectomy and posterior fixation, s/p corpectomy and anterior interbody fixation due pathologic fracture T2 (Osteosarcoma), s/p Radiotherapy.
Findings:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Swelling and T2 hyperintensity with enhancement in posterior back muscle and subcutaneous fat layer of thoracic level.
• Epidural extension and encroachment of both foraminal canal at T2/3.
Bone marrow signal change and enhancement in T2 and T3.
Diffuse marrow signal changes including thoracic vertebrae.
• Clinical significance indeterminate.
Conclusions:
s/p Decompressive laminectomy and posterior fixation, T1-2-3-T4-5.
s/p Anterior thoracic corpectomy and fusion T2-3.
Decreased epidural extent compared with 2021-12-16 MRI.
R/O Remained lesion, T2 and T3.
Recommendation:
Clinical co

In [26]:
print(pe_uncorrect['report'].loc[70]) #Findings에 No change~ 를 보고 improved라고 착각한듯

Exam : Whole spine MRI with contrast enhancement.  
Clinical information : RCC with multiple metastasis.      
COMPARISON : 2021-04-30 MRI.  
  
Findings :
Newly developed marrow replacing enhancing nodular lesion in the upper endplate of L2 right body. Lesion shows peripheral enhanement with internal  nonenhancement, peripheral low signal intensity rim. Schemorl`s node is suggested rather than metastasis.

Decreased peripheral enhancement of multiple marrow replacing lesions in T2 and T4, L4, right ilium and left femoral neck within scanned range.
No change of fatty marrow change in upper T- and lower L-spine to sacrum, suggestive of RTx-induced change.
 
 
Conclusion :
1. Newly devloped lesion, L2 body.
    - R/O Schmorl`s node
      - DDx. Metastasis.
2. Decreased enhancement of known multiple bone metastasis.


Recommendation : 
Clinical correlation and MRI short term follow up.


In [29]:
print(pe_uncorrect['report'].loc[93])
#
#Follow-up도 romets관련 용어라고 합니다. Conclusion에 follow-up이 나오는데, 여기에 집중할 수 있도록 해야 할 듯 합니다.

EXAM : Whole spine MRI with contrast enhancement. 
 
Clinical information : Lung cancer with brain, bone metastases. s/p intramedullary metastatic tumor removal (2022.2.7) s/p RTx on T11-L1. 
 
COMPARISON : Whole spine MRI taken on 2022.2.4. 
 
FINDINGS :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12 levels. 
T2 hyperintensity without enhancement in anterior aspect of conus medullaris at T11, T12 levels. 
- Focal pachymeningeal and leptomeningeal enhancement, subcutaneous T2 hyperintensity with enhancement at T11, T12 levels. 
No definite bone marrow replacing masses in whole spines. 
 
Central disc protrusion at T1/2, L2/3. 
Several enhancing small nodules in cerebellum and occipital lobe of cerebrum. 
R/O Brain metastases, refer to brain MRI taken on 2022.6.13. 
 
CONCLUSION :  
s/p Intramedullary tumor removal of spinal cord and laminectomy at T11, T12.
s/p Radiotherapy. 
- Focal pachymeningeal and leptomeningeal enhancement.
- Clinical significance i

### progression을 romets로 잘못 예측한 것

In [19]:
print(pe_uncorrect['report'].loc[65]) #History에 metastases를 보고 romets이라고 판단했나?



EXAM : Whole spine MRI with contrast enhancement 
HISTORY :  Prostate cancer with multiple bone metastases 
 
Compared to previous MRI taken on 2019.08.16, 
 
FINDING :  
Newly developed bone marrow replacing lesions with enhancement in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
Not well delineation of previous noted enhancing lesion in the lamina of C2. 
 
Spondylolisthesis of C4 over C5. 
Retrolisthesis of C6 over C7. 
Mild kyphotic deformity and mild central canal stenosis on C3-4-5-6-7 level. 
 
Disc bulging in T12/L1, L2/3, L3/4, and L4/5. 
- Mild central canal narrowing and bilateral neural foraminal narrowing in L3/4. 
- Bilateral moderate neural foraminal narrowing in L4/5. 
 
Spondylolytic spondylolisthesis of L5 over S1. 
- Bilateral severe neural foraminal narrowing. 
 
IMPRESSION :  
1. Probable bone metastases in T1 vertebral body, left lamina and pedicle of L5, left sacrum and left ilium.
2. Not well delineation of previous noted enhancing

### Stable을 Romets로 잘못 예측한 것.

In [24]:
print(pe_uncorrect['report'].loc[24]) #Impression에 R/O bone metastasis를 보고 romets라고 착각했을 가능성 높음.
# No~~~ change
# No evidence
# No newly developed
### 위의 No~~ 용어에 더 집중하도록 만들어야 할 것 같다.


Exam: Whole spine MRI with contrast enhancement. 
Clinical information: Breast cancer, Rt. Local recur, 
 
Compared with outside MRI taken on 2021.10.06  
 
Findings:   
No remarkable change of focal one marrow replacing and enhancing lesion in T9 vertebral body, R/O bone metastasis. 
No definite spinal cord signal change or enhancing lesion. 
No evidence of pathologic fracture in this study. 
No newly developed lesion. 
OPLL at T4-5-6.  
Disc bulging at L4/5.  
 
Impression: 
R/O Bone metastasis in T9 body without interval change. 
 
Recommendation:  
Clinical correlation.


In [26]:
# 정제된 파일로 inference 

clear_test_set = pd.read_csv('/root/pj_llm/codes/04_hallucination/error_analysis/test_dataset_para_clear.csv')

# inference using clear dataset 
# Check GPU architecture
if torch.cuda.get_device_capability()[0] >= 8:
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# Set quantization with BitsAndBytesConfig
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch_dtype,
#     bnb_4bit_use_double_quant=False,
# ) 
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 8비트 양자화 설정
    bnb_8bit_quant_type="int8",  # 양자화 타입 설정 (8비트 정수)
    bnb_8bit_compute_dtype=torch.float32,  # 연산에 사용할 데이터 타입
    bnb_8bit_use_double_quant=False,  # 추가적인 양자화 사용 여부
)

base_model_name = "ProbeMedicalYonseiMAILab/medllama3-v20"           # Hugging Face Basic Model
adapter_model_name = "/root/pj_llm/codes/04_hallucination/model_jh"
# adapter_model_name = "/root/modelfile/models/sy_original_7_len1024_4bit_fullfinetuning/model" #1순위
# adapter_model_name = "/root/modelfile/models/sy_original_7_len600_8bit-fullfinetuning" #2순위
# adapter_model_name = "/root/modelfile/models/sy_original_7_len600_4bit-fullfinetuning" #3순위
# adapter_model_name = "/root/modelfile/models/sy_original_7_len1024_4bit_earlystop" #4순위

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model.eval()

new_model = PeftModel.from_pretrained(model, adapter_model_name) #파라미터 합치는 코드
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True) #토크나이저는 베이스모델것과 동일하므로 상관없음


Unused kwargs: ['bnb_8bit_quant_type', 'bnb_8bit_compute_dtype', 'bnb_8bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `load_in_8bit_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [13]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Define the text generation pipeline
generator = pipeline("text-generation", model=new_model, tokenizer=tokenizer)

key = """Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.\n\n- Description of six labels\n    - no\n        - This label indicates the absence of any detectable abnormality or condition in the imaging study.\n        - For example, 'no metastasis' means that the imaging did not reveal any evidence of cancer spreading to other parts of the body.\n    - mets\n        - Metastasis refers to the spread of cancer cells from the primary site to other parts of the body.\n        - This label is used when imaging shows evidence of such spread, which is critical for staging and treatment planning.\n    - progression\n        - The term 'progression' is used when there is evidence that the disease is worsening or advancing.\n        - In the context of cancer, this means the tumor has grown in size or spread further since the last imaging study.\n    - stable\n        - 'Stable' indicates that there has been no significant change in the condition or findings since the previous imaging study.\n        - This means that the disease has neither progressed nor improved, remaining unchanged.\n    - improved\n        - This label signifies that there has been a positive change or reduction in the severity of the disease or abnormality.\n        - For instance, in cancer, it means the tumor size has decreased or the extent of the disease has lessened compared to prior imaging.\n    - romets\n        - 'Rule out metastasis' is used when further investigation is needed to determine if metastasis is present.\n        - It indicates that additional diagnostic tests are required to confirm or exclude the presence of metastatic disease."""

def evaluate_model(test_dataset, model, tokenizer, timestamp):
    
    def generate_response(batch):
        prompts = [f"### Instruction:\n{key}\n\n### Input:\n{input_text}\n\n### Response:" for input_text in batch['input']]
        responses = generator(prompts, max_new_tokens=10, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        generated_texts = [response[0]['generated_text'].replace(prompt, "").strip() for response, prompt in zip(responses, prompts)]
        return {'predicted': generated_texts}

    test_dataset = test_dataset.map(generate_response, batched=True, batch_size=1, desc="Processing Dataset")  # Adjust batch_size as needed

    correct = test_dataset['output']
    predicted = test_dataset['predicted']

    # Define the regular expression pattern to match the specified labels
    pattern = re.compile(r'\b(?:no|mets|progression|stable|improved|romets)\b')  # Regular expression pattern
    
    # Filter the predicted labels using the regex pattern
    filtered = [(c, pattern.search(p).group(0) if pattern.search(p) else p) for c, p in zip(correct, predicted)]  # Using regex to filter and extract labels
    print('Was filtering successful? - ', 'YES' if len(filtered)==len(correct) else 'NO')

    result_save_path = '/root/pj_llm/codes/04_hallucination/error_analysis/para_clear_results'
    # Ensure the directory for metric exists before saving
    metric_path = f'{result_save_path}/metrics/{timestamp}'
    os.makedirs(metric_path+'/', exist_ok=True)

    # Save filtered correct and predicted labels to a CSV file 
    filtered_df = pd.DataFrame(filtered, columns=['correct', 'predicted'])  # Creating dataframe with filtered data
    results_csv_path = f'{metric_path}/results_{timestamp}.csv'
    filtered_df.to_csv(results_csv_path, index=False)

    # LabelEncoder
    le = LabelEncoder()
    le.fit([c for c, _ in filtered] + [p for _, p in filtered])  # Using filtered values for encoding
    true_labels = le.transform([c for c, _ in filtered])
    predicted_labels = le.transform([p for _, p in filtered])

    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, average='macro')
    recall = recall_score(true_labels, predicted_labels, average='macro')
    f1 = f1_score(true_labels, predicted_labels, average='macro')

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

    # Write metric result to an json file
    metrics_filename = f'{metric_path}/metrics_{timestamp}.json'
    with open(metrics_filename, 'w') as f:
        json.dump(metrics, f)

    print(f'Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')

# Evaluate the model
# evaluate_model(clear_test_set, new_model, tokenizer, timestamp)
evaluate_model(test_dataset, new_model, tokenizer, timestamp)


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM'

TypeError: evaluate_model.<locals>.generate_response() got an unexpected keyword argument 'batched'

In [21]:
########### 추론하기 ############
key = "Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.\n\n- Description of six labels\n    - 'no'\n        - a state of being unresponsive to treatment\n        - May mean cancer remains intact or remains unchanged\n    - 'mets'\n        - a state in which a transition has occurred\n        - If the cancer has spread from its original location to another site\n    - 'progression'\n        - a state of progression of cancer\n        - If the cancer has grown or worsened\n    - 'stable'\n        - a stable condition of cancer\n        - If the cancer is no longer progressing and remains in place\n    - 'improved'\n        - a state of improvement in cancer\n        - If the size or condition of the cancer improves\n    - 'romets'\n        - a state of reduced transition\n        - If the original metastatic cancer is reduced\n"
result_save_path = '/root/modelfile/'

generator = pipeline("text-generation", model=new_model, tokenizer=tokenizer) #new_model: 파라미터 합친 모델

def evaluate_model(test_dataset, model, tokenizer):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    def generate_response(batch):
        print(batch)
        prompts = [f"### Instruction:\n{key}\n\n### Input:\n{input_text}\n\n### Instruction:\n{key}\n\n### Response:" for input_text in batch['input']]
        responses = generator(prompts, max_new_tokens=5, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        generated_texts = [response[0]['generated_text'].replace(prompt, "").strip() for response, prompt in zip(responses, prompts)]
        return {'predicted': generated_texts}

    # print(test_dataset)
    test_dataset = test_dataset.map(generate_response)  # Adjust batch_size as needed

    correct = test_dataset['output']
    predicted = test_dataset['predicted']

    # Define the regular expression pattern to match the specified labels
    pattern = re.compile(r'\b(?:no|mets|progression|stable|improved|romets)\b')  # Regular expression pattern

    # Filter the predicted labels using the regex pattern
    filtered = [(c, pattern.search(p).group(0)) for c, p in zip(correct, predicted) if pattern.search(p)]  # Using regex to filter and extract labels
    print('filtered:', filtered)
    
    # Save filtered correct and predicted labels to a CSV file
    filtered_df = pd.DataFrame(filtered, columns=['correct', 'predicted'])  # Creating dataframe with filtered data
    results_csv_path = f'{result_save_path}/result/results_{timestamp}.csv'
    filtered_df.to_csv(results_csv_path, index=False)
    
    # LabelEncoder
    le = LabelEncoder()
    le.fit([c for c, _ in filtered] + [p for _, p in filtered])  # Using filtered values for encoding
    true_labels = le.transform([c for c, _ in filtered])
    predicted_labels = le.transform([p for _, p in filtered])

    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, average='macro')
    recall = recall_score(true_labels, predicted_labels, average='macro')
    f1 = f1_score(true_labels, predicted_labels, average='macro')

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

    # Save metrics to a unique file
    metrics_filename = f'{result_save_path}/result/metrics_{timestamp}.json'
    with open(metrics_filename, 'w') as f:
        json.dump(metrics, f)

    print(f'Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')

# Evaluate the model
evaluate_model(test_dataset, model, tokenizer)

OSError: ProbeMedicalYonseiMAILab/medllama3-v20-finetuned is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`